# Running IPCA + RFF geometric analysis of complexity for 500 stocks and 30 years data

## Loading data

In [1]:
#imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import missingno as msno
import itertools

import numpy as np
from matplotlib import pyplot as plt

import autograd.numpy as anp
from pymanopt import Problem
from pymanopt.manifolds import Grassmann
from pymanopt.function import autograd as pymanopt_autograd
from pymanopt.optimizers import ConjugateGradient, SteepestDescent, TrustRegions

import requests
import zipfile
import io
import os
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
for p in [ROOT_DIR, ROOT_DIR / "src"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.rff import RandomFourierFeatures


In [2]:
# load data from ipca_char_data_filtered.parquet

DATA_DIR = ROOT_DIR / "data"
df_char_filtered = pd.read_parquet(DATA_DIR / "ipca_char_data_filtered.parquet")

In [3]:
df_char_filtered.head()

,permno,yyyymm,AM,AbnormalAccruals,AnnouncementReturn,AssetGrowth,BMdec,Beta,BetaFP,BetaLiquidityPS,...,VolumeTrend,betaVIX,cfp,dNoa,hire,Price,Size,STreversal,excess_ret,y_ipca
6359,10104,1994-01-01,0.127079,0.116246,-0.118802,-0.239069,0.133073,1.985119,2.026764,-0.295296,...,-0.011868,0.021741,0.034075,0.108471,-0.124892,-3.469635,-9.139615,-0.117391,0.117391,0.027237
6360,10104,1994-02-01,0.125607,0.116246,-0.118802,-0.239069,0.133073,1.972944,2.008414,-0.284722,...,-0.011071,-0.000676,0.033680,0.108471,-0.124892,-3.496508,-9.151264,-0.027237,0.027237,-0.026515
6361,10104,1994-03-01,0.129029,0.116246,0.005112,-0.239069,0.133073,1.951014,2.014694,-0.301786,...,-0.010681,-0.012093,0.034598,0.108471,-0.124892,-3.469635,-9.124391,0.026515,-0.026515,-0.070039
6362,10104,1994-04-01,0.138746,0.116246,0.005112,-0.239069,0.133073,1.946702,2.041135,-0.179182,...,-0.010495,-0.001282,0.037203,0.108471,-0.124892,-3.397022,-9.051779,0.070039,-0.070039,0.146444
6363,10104,1994-05-01,0.120721,0.116246,0.005112,-0.239069,0.133073,1.935432,2.041004,-0.186885,...,-0.010062,0.011399,0.032370,0.108471,-0.124892,-3.533687,-9.190940,-0.146444,0.146444,0.094891


In [4]:
# check if any column has nan/missin values
df_char_filtered.isna().sum()
# only print the columns with missing values
print(df_char_filtered.isna().sum()[df_char_filtered.isna().sum() > 0])
#check if these columns with missing values are important for our analysis, if not we can drop them
missing_columns = df_char_filtered.isna().sum()[df_char_filtered.isna().sum() > 0].index
#print(df_char_filtered[missing_columns].head())
#print the dates in whch these columns have missing values
#print(df_char_filtered[df_char_filtered[missing_columns].isna().any(axis=1)].yyyymm.unique())
# find the longest consecutive sequence of missing values in these columns(streak in terms of date)
for col in missing_columns:
    is_missing = df_char_filtered[col].isna()
    max_streak = 0
    current_streak = 0
    for missing in is_missing:
        if missing:
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0
    print(f"Column: {col}, Max Consecutive Missing Values: {max_streak}")
#show the number of missing values per permno
missing_by_permno = df_char_filtered.groupby("permno")[missing_columns].apply(lambda x: x.isna().sum())
#show in the order of decreasing number of missing values
missing_by_permno = missing_by_permno.sort_values(by=missing_columns.tolist(), ascending=False)
print(missing_by_permno.head(20))  

excess_ret    403
y_ipca        476
dtype: int64
Column: excess_ret, Max Consecutive Missing Values: 85
Column: y_ipca, Max Consecutive Missing Values: 85
        excess_ret  y_ipca
permno                    
81593           85      86
10693           44      44
21020           24      25
11896           14      14
84723            5       5
90090            3       3
81061            2       2
81138            2       2
81857            2       2
82298            2       2
82486            2       2
82618            2       2
82643            2       2
82686            2       2
82759            2       2
83111            2       2
83435            2       2
83906            2       2
84597            2       2
84761            2       2


In [5]:
#Lets drop the top4 and fill the others with previous values
permnos_to_drop = missing_by_permno.head(4).index
df_char_filtered = df_char_filtered[~df_char_filtered.permno.isin(permnos_to_drop)]
df_char_filtered[missing_columns] = df_char_filtered[missing_columns].fillna(method="ffill")


/var/folders/js/pgf30x597b53rt2mghx7jpmm0000gn/T/ipykernel_68906/3808254857.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_char_filtered[missing_columns] = df_char_filtered[missing_columns].fillna(method="ffill")


In [6]:
#Now lets check if there are any missing values left
print(df_char_filtered.isna().sum()[df_char_filtered.isna().sum() > 0])

Series([], dtype: int64)


In [7]:
#print the number of columns
print(f"Number of columns: {df_char_filtered.shape[1]}")

Number of columns: 78


## Import parallel sweep helpers from src._grass_worker

In [8]:
import sys
import importlib
from pathlib import Path
import types

from src._grass_worker import (
    build_rff_inputs, build_jobs, run_sweep, aggregate, make_results_df,
    Z_VALUES, _z_label,
    GAMMA, WINDOW_LEN, N_FEATURES_RFF, NUM_FACTORS_LIST, NUM_ITER_RFF,
)
# Ensure src/ is on sys.path so top-level modules in src can be imported
src_dir = ROOT_DIR / "src" #this seems to be the only way to reliably import from src/ in both .ipynb and .py contexts without causing import errors in one or the other
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Try to import the real module; if missing, install a lightweight shim to avoid ImportError
try:
    backtest_GRASS_IPCA = importlib.import_module("backtest_GRASS_IPCA")
except Exception:
    shim = types.ModuleType("backtest_GRASS_IPCA")
    def run_backtest(*args, **kwargs):
        raise RuntimeError("backtest_GRASS_IPCA.run_backtest is not available in this environment.")
    shim.run_backtest = run_backtest
    # add any other expected symbols here as simple stubs if you discover they're required
    sys.modules["backtest_GRASS_IPCA"] = shim
    backtest_GRASS_IPCA = shim
    print("Warning: using shim for backtest_GRASS_IPCA (real module not found).")

## Prepare panel: MultiIndex (date, permno), lag characteristics, rename target

In [ ]:
# Convert yyyymm → datetime, set MultiIndex (date, permno)
df = df_char_filtered.copy()
df["date"] = pd.to_datetime(df["yyyymm"])
df = df.drop(columns=["yyyymm"]).set_index(["date", "permno"]).sort_index()

# Characteristic columns (all except the return target)
char_cols = [c for c in df.columns if c != "y_ipca"]
print(f"Characteristic columns ({len(char_cols)}): {char_cols[:6]} ...")

# Lag characteristics 1 period within each permno (Z_{t-1} predicts r_t)
df[char_cols] = df[char_cols].groupby(level="permno").shift(1)

# Rename return column to the convention expected by run_ipca_grass_v2
df = df.rename(columns={"y_ipca": "ret"})
df = df.dropna(subset=["ret"])

# build_rff_inputs expects df_base to have a "Price" column alongside "ret";
# it is not used by the backtest — add a dummy so the API stays unchanged.
df["Price"] = 1.0

print(f"Panel shape   : {df.shape}")
print(f"Date range    : {df.index.get_level_values('date').min().date()} → "
      f"{df.index.get_level_values('date').max().date()}")
print(f"Unique permnos: {df.index.get_level_values('permno').nunique()}")


Characteristic columns (75): ['AM', 'AbnormalAccruals', 'AnnouncementReturn', 'AssetGrowth', 'BMdec', 'Beta'] ...
Panel shape   : (170070, 76)
Date range    : 1994-01-01 → 2024-12-01
Unique permnos: 496


## RFF expansion of characteristics

In [10]:
# ===========================================================================
# 1. (OPTIONAL) OVERRIDE GRID DEFAULTS
#    Uncomment / edit any line to override the defaults from _grass_worker.py
# ===========================================================================
N_FEATURES_RFF   = [256, 1024, 4096]   # low/mid/high — captures P-scaling cleanly
NUM_FACTORS_LIST = [4, 8, 16, 32]      # brackets the expected k₀ for 500 stocks
NUM_ITER_RFF     = 3                   # keep at 3, enough to average out RFF noise
WINDOW_LEN       = 12                  # keep as is
GAMMA            = 0.25                # keep as is
Z_VALUES         = [0, 10, 100]        # zero (illusory baseline), sweet spot, over-regularised       # 0.0 = OLS baseline; add e.g. 1e-2, 1.0 for shrinkage

# ===========================================================================
# 2. PREPARE INPUT DATA
#    Dense char matrix for RFF — impute cross-sectionally, then col-wise fallback
# ===========================================================================
date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)        # ensure float64 — required for np.sin/cos in RFF
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")


/Users/chaithanyapakala/Desktop/researchAndrew/TheVirtueOfComplexity_PaperReplication_Experimentation/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/chaithanyapakala/Desktop/researchAndrew/TheVirtueOfComplexity_PaperReplication_Experimentation/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/chaithanyapakala/Desktop/researchAndrew/TheVirtueOfComplexity_PaperReplication_Experimentation/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/chaithanyapakala/Desktop/researchAndrew/TheVirtueOfComplexity_PaperReplication_Experimentation/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean 

X_chars shape : (170070, 75)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [11]:
X_chars

AM  AbnormalAccruals  AnnouncementReturn  \
date       permno                                                   
1994-01-01 10104   0.825544         -0.007795            0.003087   
           10107   0.825544         -0.007795            0.003087   
           10138   0.825544         -0.007795            0.003087   
           10145   0.825544         -0.007795            0.003087   
           10147   0.825544         -0.007795            0.003087   
...                     ...               ...                 ...   
2024-12-01 91068   0.095964         -0.011654           -0.014940   
           91152   0.283445         -0.126017           -0.025517   
           91233   0.087452         -0.071839            0.014583   
           91556   0.279588         -0.025738            0.016115   
           92655   0.487430          0.002143           -0.014940   

                   AssetGrowth     BMdec      Beta    BetaFP  BetaLiquidityPS  \
date       permno                                                               
1994-01-01 10104     -0.068889  0.368755  0.751005  0.969723        -0.004856   
           10107     -0.068889  0.368755  0.751005  0.969723        -0.004856   
           10138     -0.068889  0.368755  0.751005  0.969723        -0.004856   
           10145     -0.068889  0.368755  0.751005  0.969723        -0.004856   
           10147     -0.068889  0.368755  0.751005  0.969723        -0.004856   
...                        ...       ...       ...       ...              ...   
2024-12-01 91068     -0.161221  0.050208  0.895615  1.377417        -0.100633   
           91152     -0.102888 -0.024251  1.156392  1.198892         0.037090   
           91233     -0.096168  0.018390  0.748228  0.930522        -0.103502   
           91556     -0.065863  0.108767  0.799962  1.059803        -0.102801   
           92655     -0.114019  0.188475  0.347785  1.038234        -0.038872   

                   BidAskSpread  BookLeverage  ...       VolSD  VolumeTrend  \
date       permno                              ...                            
1994-01-01 10104       0.005727     -2.264936  ...   -9.749340    -0.005768   
           10107       0.005727     -2.264936  ...   -9.749340    -0.005768   
           10138       0.005727     -2.264936  ...   -9.749340    -0.005768   
           10145       0.005727     -2.264936  ...   -9.749340    -0.005768   
           10147       0.005727     -2.264936  ...   -9.749340    -0.005768   
...                         ...           ...  ...         ...          ...   
2024-12-01 91068       0.007740     -2.552699  ... -111.086357     0.000923   
           91152       0.006426     14.716286  ...   -1.360742     0.017554   
           91233       0.004816     -5.816388  ...  -16.099022     0.014013   
           91556       0.006287     -2.821890  ...  -16.076866     0.001589   
           92655       0.004771     -2.982447  ...  -15.730351     0.001424   

                    betaVIX       cfp      dNoa      hire  Price       Size  \
date       permno                                                             
1994-01-01 10104  -0.000071  0.072224 -0.028052 -0.025607    1.0  -9.229930   
           10107  -0.000071  0.072224 -0.028052 -0.025607    1.0  -9.229930   
           10138  -0.000071  0.072224 -0.028052 -0.025607    1.0  -9.229930   
           10145  -0.000071  0.072224 -0.028052 -0.025607    1.0  -9.229930   
           10147  -0.000071  0.072224 -0.028052 -0.025607    1.0  -9.229930   
...                     ...       ...       ...       ...    ...        ...   
2024-12-01 91068   0.008655  0.021276 -0.089222 -0.100531    1.0 -11.336507   
           91152  -0.005091  0.019516 -0.069089 -0.073579    1.0 -11.162722   
           91233   0.004495  0.024681 -0.028200 -0.110585    1.0 -13.092697   
           91556   0.002658  0.049162 -0.022481 -0.066986    1.0 -10.842460   
           92655  -0.004177  0.051763 -0.056092 -0.095238    1.0 -13.238469   

                 

## Rolling IPCA + RFF parallel sweep

Mirrors the cell-40 pattern from `VOC_everywhere.ipynb`:
1. `build_rff_inputs` — pre-compute all (n_feat × seed) RFF DataFrames
2. `build_jobs` — build full (k, P, z, seed) Cartesian job list, sorted longest-first (LPT)
3. `run_sweep` — dispatch in parallel via joblib/loky
4. `aggregate` + `make_results_df` — average over seeds → MultiIndex summary

## Run with 496 stocks, kernel is crashing

In [11]:
# ===========================================================================
# 3. RUN PIPELINE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,          # must contain ["Price", "ret"]
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,   # already imported in cell 2
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)


Pre-computing RFF features ...
  9 RFF datasets ready.


: 

### Results — plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"IPCA + RFF on 500 stocks (1994-2025)\n"
    f"window_len={WINDOW_LEN} months, gamma={GAMMA}, z={Z_VALUES}",
    fontsize=13,
)

metrics = [
    ("avg_r2_oos",               "OOS R²"),
    ("avg_sharpe",               "Sharpe (annualised)"),
    ("avg_subspace_stability",   "Mean d_proj  (↓ better)"),
    ("avg_erank",                "Mean erank(Σ̂_f)  (↑ better)"),
    ("avg_spectral_gap",         "Spectral gap ratio  (↑ better)"),
    ("avg_max_principal_angle",  "Max principal angle rad  (↓ better)"),
]

for ax, (col, title) in zip(axes.flat, metrics):
    for nf in NUM_FACTORS_LIST:
        subset = results_rff_df.xs(nf, level="k")[col]
        ax.plot(subset.index.get_level_values("P"), subset.values,
                marker="o", label=f"k={nf}")
    ax.set_title(title)
    ax.set_xlabel("N_FEATURES_RFF  (P)")
    ax.set_xscale("log")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()


## Run with top50 stocks

### Loading top50(mkt cap) stocks

In [19]:
import wrds
import pandas as pd

db = wrds.Connection()

# Get the 496 permnos already in your panel
panel_permnos = df_char_filtered["permno"].unique().tolist()
permnos_str = ",".join(map(str, panel_permnos))

sql = f"""
    SELECT
        msf.permno,
        msf.date,
        ABS(msf.prc) * msf.shrout AS me
    FROM crsp.msf
    JOIN crsp.msenames AS names
      ON msf.permno = names.permno
     AND msf.date BETWEEN names.namedt AND names.nameendt
    WHERE msf.date    >= '1994-01-01'
      AND msf.date    <= '2025-12-31'
      AND msf.permno  IN ({permnos_str})
"""
df_me = db.raw_sql(sql, date_cols=["date"])
db.close()

top50_permnos = (
    df_me.groupby("permno")["me"]
    .mean()
    .nlargest(50)
    .index.tolist()
)

df_top50 = df_char_filtered[df_char_filtered["permno"].isin(top50_permnos)]
print(f"Stocks: {len(top50_permnos)}, Panel shape: {df_top50.shape}")



WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Stocks: 50, Panel shape: (17803, 78)


In [21]:
import wrds

db = wrds.Connection()

permnos_str = ",".join(map(str, top50_permnos))

name_map = db.raw_sql(f"""
    SELECT DISTINCT ON (permno) permno, ticker, comnam
    FROM crsp.msenames
    WHERE permno IN ({permnos_str})
    ORDER BY permno, namedt DESC   -- most recent name entry
""")

print(name_map.sort_values("comnam").to_string(index=False))


WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
 permno ticker                           comnam
  22592    MMM                            3M CO
  66093      T                      A T & T INC
  20482    ABT              ABBOTT LABORATORIES
  90319  GOOGL                     ALPHABET INC
  13901     MO                 ALTRIA GROUP INC
  84788   AMZN                   AMAZON COM INC
  59176    AXP              AMERICAN EXPRESS CO
  66800    AIG AMERICAN INTERNATIONAL GROUP INC
  14008   AMGN                        AMGEN INC
  14593   AAPL                        APPLE INC
  59408    BAC             BANK OF AMERICA CORP
  83443    BRK       BERKSHIRE HATHAWAY INC DEL
  17778    BRK       BERKSHIRE HATHAWAY INC DEL
  19561     BA                        BOEING CO
  19393    BMY          BRISTOL MYERS SQUIBB CO
  14541    CVX                 CHEVRON CORP NEW
  76076   CSCO            

In [22]:
name_map


,permno,ticker,comnam
0,10104,ORCL,ORACLE CORP
1,10107,MSFT,MICROSOFT CORP
2,11308,KO,COCA COLA CO
3,11850,XOM,EXXON MOBIL CORP
4,12060,GE,GENERAL ELECTRIC CO
5,12490,IBM,INTERNATIONAL BUSINESS MACHS COR
6,13856,PEP,PEPSICO INC
7,13901,MO,ALTRIA GROUP INC
8,14008,AMGN,AMGEN INC
9,14541,CVX,CHEVRON CORP NEW


In [ ]:
#lets see if there are any missing nans in df_top50
print(df_top50.isna().sum()[df_top50.isna().sum() > 0])

Series([], dtype: int64)


### Simulate with top50 stocks

In [26]:
# prepare panel like above for the top50 stocks and run the backtest on this smaller universe to see if we can get better convergence with more RFF features and more iterations
df = df_top50.copy()
df["date"] = pd.to_datetime(df["yyyymm"])
df = df.drop(columns=["yyyymm"]).set_index(["date", "permno"]).sort_index()

# Characteristic columns (all except the return target)
char_cols = [c for c in df.columns if c != "y_ipca"]
print(f"Characteristic columns ({len(char_cols)}): {char_cols[:6]} ...")

# Lag characteristics 1 period within each permno (Z_{t-1} predicts r_t)
df[char_cols] = df[char_cols].groupby(level="permno").shift(1)

# Rename return column to the convention expected by run_ipca_grass_v2
df = df.rename(columns={"y_ipca": "ret"})
df = df.dropna(subset=["ret"])

# build_rff_inputs expects df_base to have a "Price" column alongside "ret";
# it is not used by the backtest — add a dummy so the API stays unchanged.
df["Price"] = 1.0

print(f"Panel shape   : {df.shape}")
print(f"Date range    : {df.index.get_level_values('date').min().date()} → "
      f"{df.index.get_level_values('date').max().date()}")
print(f"Unique permnos: {df.index.get_level_values('permno').nunique()}")

Characteristic columns (75): ['AM', 'AbnormalAccruals', 'AnnouncementReturn', 'AssetGrowth', 'BMdec', 'Beta'] ...
Panel shape   : (17803, 76)
Date range    : 1994-01-01 → 2024-12-01
Unique permnos: 50


In [ ]:
# these will have one nan per columns, as we shifted to make Z_t-1) predict r_t, so this is expected


AM                    50
AbnormalAccruals      50
AnnouncementReturn    50
AssetGrowth           50
BMdec                 50
                      ..
dNoa                  50
hire                  50
Size                  50
STreversal            50
excess_ret            50
Length: 74, dtype: int64


In [32]:
# ===========================================================================
# 1. (OPTIONAL) OVERRIDE GRID DEFAULTS
#    Uncomment / edit any line to override the defaults from _grass_worker.py
# ===========================================================================
N_FEATURES_RFF   = [1024]   # low/mid/high — captures P-scaling cleanly
NUM_FACTORS_LIST = [24]      # brackets the expected k₀ for 500 stocks
NUM_ITER_RFF     = 3                   # keep at 3, enough to average out RFF noise
WINDOW_LEN       = 12                  # keep as is
GAMMA            = 0.25                # keep as is
Z_VALUES         = [10]        # zero (illusory baseline), sweet spot, over-regularised       # 0.0 = OLS baseline; add e.g. 1e-2, 1.0 for shrinkage

# ===========================================================================
# 2. PREPARE INPUT DATA
#    Dense char matrix for RFF — impute cross-sectionally, then col-wise fallback
# ===========================================================================
date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)        # ensure float64 — required for np.sin/cos in RFF
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")

/Users/chaithanyapakala/Desktop/researchAndrew/TheVirtueOfComplexity_PaperReplication_Experimentation/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/chaithanyapakala/Desktop/researchAndrew/TheVirtueOfComplexity_PaperReplication_Experimentation/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/chaithanyapakala/Desktop/researchAndrew/TheVirtueOfComplexity_PaperReplication_Experimentation/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/chaithanyapakala/Desktop/researchAndrew/TheVirtueOfComplexity_PaperReplication_Experimentation/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean 

X_chars shape : (17803, 75)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [ ]:
# ===========================================================================
# 3. RUN PIPELINE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,          # must contain ["Price", "ret"]
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,   # already imported in cell 2
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)

Pre-computing RFF features ...
  3 RFF datasets ready.
Dispatching 3 jobs across 3 workers (cost range 120397–120397, LPT order) ...


[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.
